# 1. From a CSV to model-ready days

This notebook uses fabricated HR. Nothing here estimates real prediction performance. Run from the project environment; every step uses the same functions as the app.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if ROOT.name != 'HEARTBEAT_LAB':
    ROOT = ROOT / 'HEARTBEAT_LAB'
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import pandas as pd
from heartbeat_lab import prepare_days, allocate_customers, customer_cv


## Read the file

One person per CSV. The simplest schema is `timestamp,hr`, with explicit timezone offsets. Native epoch CSVs also work. Do not treat UTC time as local clock time.

In [ ]:
frame = pd.read_csv(ROOT / 'examples/synthetic_hr.csv')
frame.head()

## Apply the fixed rules

Keep HR 25–230 bpm. An adequate local day has ≥8 observed clock-hours AND (≥60 events OR ≥8 hours interval-union coverage). Point observations have zero interval coverage. This is the day gate, not the original longitudinal cohort gate.

In [ ]:
days, hr, quality = prepare_days(frame)
quality

## Inspect daily summaries and missingness

Each day has 288 five-minute clock bins. Multiple events in a bin are averaged; empty bins remain NaN. The eight daily summary statistics are based on valid events, not filled bins.

In [ ]:
display(days)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(np.arange(288)/12, hr[0])
ax.set(xlabel='Local clock hour', ylabel='HR (bpm)', title='First synthetic day; gaps are missing bins')
plt.show()

## Keep missingness visible

The model fills missing bins using training customers only. Uploads must never estimate or update those fill values. Retain the observation mask for summary features and the quality display.

In [ ]:
availability = pd.DataFrame({'date': days.date, 'observed_bins': np.isfinite(hr).sum(axis=1)})
availability

## Next

Notebook 2 follows customer allocation, fitting and model comparison. For your own unlabelled file, replace only the read_csv path above; inference needs a separately trained bundle.